# DocMind RAG evaluation

This notebook explains how DocMind evaluates retrieval and generated answers. It covers the DeepEval pipeline in `backend/eval/`, synthetic goldens, BEIR/Scifact, metric interpretation, reproducible testing, cost control, and the important distinction between LLM-judged contextual metrics and exact IR metrics such as Recall@k.

## 1. What should be evaluated?

A RAG system has at least two separable failure modes:

1. **Retrieval failure**: the needed evidence was not found or was ranked too low. A perfect LLM cannot answer reliably without context.
2. **Generation failure**: the evidence was retrieved but the answer is wrong, incomplete, off-topic, or unsupported.

Evaluate both. An answer that sounds good does not prove the retriever works; conversely, a low answer score may be caused by poor generation despite excellent retrieval. Agentic RAG adds a third layer: the planner/tool loop must be evaluated for step count, repeated actions, latency, and recovery after insufficient evidence.

## 2. Evaluation data sources

### Synthetic document QA

DeepEval's `Synthesizer` creates question/expected-answer goldens from local documents in `docs/` or `data/uploads/`. The generated goldens are saved to `eval/synthetic_goldens.json` so later runs reuse exactly the same questions. This is useful for fast functional tests, but generated questions can be overly easy or stylistically similar to the source. Manually review a representative sample.

### BEIR Scifact

BEIR Scifact is a public scientific-information-retrieval benchmark. It provides a corpus, short claims/queries, and qrels (known relevant document IDs). It is excellent for retrieval benchmarking, but not a natural QA reference-answer dataset. Treat answer-generation metrics on BEIR as secondary; exact qrels retrieval metrics should be the primary result.

### Human-curated holdout

For production decisions, create a frozen set of real user questions with independently reviewed expected answers and relevant chunk/document IDs. Split it into development, validation, and final test partitions. Do not tune chunking or prompts against your final holdout set.

In [ ]:
from pathlib import Path
import json
goldens_path = Path('eval/synthetic_goldens.json')
if goldens_path.exists():
    goldens = json.loads(goldens_path.read_text(encoding='utf-8'))
    print(f'{len(goldens)} cached synthetic goldens')
    display(goldens[:2])
else:
    print('No cached goldens yet. Run the synthetic evaluation cell below.')


## 3. The metrics in this project

DocMind runs these DeepEval LLM-judged metrics with `EVAL_THRESHOLD` (default `0.7`):

| Metric | Question it answers | High score means |
|---|---|---|
| Contextual Precision | Are relevant context chunks ranked before irrelevant ones? | useful context is near the top |
| Contextual Recall | Does retrieved context contain the evidence needed by expected output? | evidence is present |
| Contextual Relevancy | How much of all supplied context is directly relevant to the input? | little context noise |
| Faithfulness | Is the actual answer supported by retrieved context? | low hallucination risk |
| Answer Relevancy | Does the actual answer address the question? | answer is on-topic |

DeepEval scores are model-judged estimates, not ground truth. They can vary by judge model and prompt. Persist the judge model/version, threshold, RAG model, corpus revision, chunk parameters, and random seed/configuration with reports.

## 4. Exact information-retrieval metrics

DeepEval **ContextualPrecisionMetric is not mathematical Precision@k**, and ContextualRecallMetric is not exact Recall@k. Exact IR scores require labelled relevance judgments (qrels): the set of document or chunk IDs known to be relevant to each query.

For a query, let `R` be the set of relevant IDs and `S_k` be the first `k` retrieved IDs:

```text
Precision@k = |R ∩ S_k| / k
Recall@k    = |R ∩ S_k| / |R|
```

For example, with gold `{A, B}` and top-5 `{A, C, D, E, F}`:

```text
Precision@5 = 1 / 5 = 0.20
Recall@5    = 1 / 2 = 0.50
```

Also report MRR (rank of first relevant result) and nDCG when relevance has grades. For overlapping chunks, report both strict chunk-level and more forgiving document-level values.

In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    retrieved = set(retrieved_ids[:k])
    return len(retrieved & set(relevant_ids)) / k

def recall_at_k(retrieved_ids, relevant_ids, k):
    relevant = set(relevant_ids)
    return 0.0 if not relevant else len(set(retrieved_ids[:k]) & relevant) / len(relevant)

gold = ['A', 'B']
ranked = ['A', 'C', 'D', 'E', 'F']
print('Precision@5:', precision_at_k(ranked, gold, 5))
print('Recall@5:   ', recall_at_k(ranked, gold, 5))


## 5. How the DocMind evaluation pipeline works

`run_evaluation(mode, max_samples)` performs these operations:

1. Rebuilds the process-local sparse index from mounted `data/uploads/` and `docs/` files.
2. In BEIR mode, downloads/loads the 5,183-document Scifact corpus and indexes it.
3. Loads test cases: cached/generated synthetic goldens, Scifact qrels-derived cases, or both.
4. For each case, retrieves the configured `EVAL_K` chunks (default 5).
5. Generates an answer through DocMind.
6. Builds a DeepEval `LLMTestCase(input, actual_output, expected_output, retrieval_context)`.
7. Runs the five metrics and returns a JSON-ready report with mean score, threshold, pass/fail, and number evaluated.

The evaluator executes synchronously and disables DeepEval disk caching as a Windows compatibility workaround. Synthetic generation and DeepEval judging can call external LLMs; a run therefore has cost and rate-limit implications.

## 6. First controlled synthetic run

Before running this cell, set safe values in `.env` or the environment:

```env
EVAL_K=3
EVAL_THRESHOLD=0.7
DEEPEVAL_MODEL=gpt-4o-mini
```

Use only one or a few examples first. The first run may generate goldens and load embedding models. If `eval/synthetic_goldens.json` already exists, it is reused. Delete it only when you intentionally want newly generated questions.

In [ ]:
import os
os.environ['EVAL_K'] = '3'
os.environ['EVAL_THRESHOLD'] = '0.7'
# Choose an accessible, lower-cost judge model appropriate for your provider.
os.environ.setdefault('DEEPEVAL_MODEL', 'gpt-4o-mini')

from backend.eval.pipeline import run_evaluation
report = run_evaluation(mode='synthetic', max_samples=1)
report


## 7. Reading a report

A result has the following shape:

```json
{
  "threshold": 0.7,
  "samples": 5,
  "k": 3,
  "metrics": {
    "FaithfulnessMetric": {"score": 0.91, "passed": true, "threshold": 0.7, "evaluated": 5}
  }
}
```

If contextual recall and faithfulness are high but contextual relevancy is low, the answer can be correct while too many irrelevant chunks were given to the model. Try a smaller `EVAL_K`, better chunking, a reranker, metadata filtering, or document-level query routing. Do not claim a precision improvement solely from a single small run; compare the same frozen goldens and record confidence/variance across runs.

In [ ]:
for metric, value in report.get('metrics', {}).items():
    score = value['score']
    print(f"{metric:30} score={score if score is not None else 'n/a'}  passed={value['passed']}")


## 8. BEIR / Scifact run

Run a single sample first. The first BEIR execution must index the full Scifact corpus even if `max_samples=1`; this is why it may take minutes. The HuggingFace data cache makes later downloads faster, but the current process-local BM25 index is rebuilt when a new process starts.

BEIR qrels identify relevant source documents. The current DeepEval adapter converts their relevant abstracts into expected text. This is useful for context-quality inspection, but a complete benchmark implementation should additionally map returned chunk sources back to Scifact document IDs and compute exact Recall@k, Precision@k, MRR, and nDCG against qrels.

In [ ]:
# Requires internet on first run and can incur answer/judge model cost.
# beir_report = run_evaluation(mode='beir', max_samples=1)
# beir_report


## 9. Test the agentic variant fairly

Classic and agentic systems must receive the same frozen questions and documents. Compare:

- exact document/chunk Recall@1, @3, @5
- contextual recall/precision/relevancy
- faithfulness and answer relevancy
- answer latency and token cost
- agent iterations and tool calls
- web-search usage and source provenance
- clarification rate
- repeated-action rate

Agentic RAG has extra cost and latency. It is worthwhile only if it improves evidence coverage and answer quality for complex/multi-hop questions without degrading simple lookup behavior. Keep classic RAG as a baseline and route only appropriate question types to the agent.

In [ ]:
# API smoke test after the FastAPI server is running:
# import requests
# response = requests.post('http://localhost:8000/query', json={
#     'question': 'Compare Project Alpha and Project Beta production launch dates and owners.',
#     'mode': 'agentic', 'max_iterations': 4, 'top_k': 5,
# })
# response.json()


## 10. Reproducibility, cost, and troubleshooting

- Start with `max_samples=1`, `EVAL_K=3`, and a low-cost judge model.
- Save terminal JSON to `eval/` and preserve `synthetic_goldens.json`.
- Do not regenerate goldens during every run; it changes the benchmark.
- DeepEval can hit token-per-minute limits when metrics run on many long contexts. Reduce samples/k, select a smaller judge, and wait for the rate-limit window.
- HuggingFace Windows symlink and `hf_xet` notices are performance/storage warnings, not failed downloads.
- Chroma telemetry messages are optional analytics compatibility warnings; they are unrelated to retrieval correctness.
- Review per-case failures, not only averages. A 1.0 recall average on easy synthetic questions may conceal failures on ambiguous or adversarial questions.

Useful commands:

```powershell
python cli.py eval --mode synthetic --max-samples 5 | Tee-Object eval\synthetic_report.txt
python cli.py eval --mode beir --max-samples 1 | Tee-Object eval\beir_report.txt
python -m pytest -q
```
